In [ ]:
import io
import pandas as pd
import requests
import datetime
from typing import List
from sqlalchemy import select, update, Table, Column, MetaData, Boolean, String, Date
%run _1_schema.ipynb

SP500_URL = "https://stockanalysis.com/list/sp-500-stocks/"
SP500_WIKI = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame: 
    # read_html sometimes returns tuple columns; flatten them into strings
    out = df.copy()
    cols: list[str] = [] #empty list
    for c in out.columns:
        if isinstance(c, tuple): # c 係咪 tuple 呢種型別 like instanceof
            c = " ".join(str(x) for x in c if x and str(x).lower() != "nan") # nan = not a number in pandas/numpy
            # " ".join() means 用 空格將多個字串合併 "market", "cap" -> market cap (string method)
        cols.append(str(c).strip())
    out.columns = cols
    return out

def pick_table_by_columns(tables: list[pd.DataFrame], required_cols: set[str]) -> pd.DataFrame:
    required_cols = set(required_cols)
    for df in tables:
        dfn = _normalize_columns(df)
        if required_cols.issubset(set(dfn.columns)): # issubset 意思係：「required_cols 入面每個欄位都喺 df.columns 裏面」
            return dfn
    # helpful error for debugging when website changes
    available = [list(_normalize_columns(df).columns) for df in tables] # list all tables columns, to see what changes
    raise ValueError(f"Cannot find table with columns {sorted(required_cols)}. Available columns: {available}")

def get_sp500_symbols_from_web() -> List[str]:
    """先試 stockanalysis，失敗就 fallback Wikipedia"""
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(SP500_URL, headers=headers, timeout=15) # if 15 seconds coudn't get data, return error
        resp.raise_for_status() # 如果 HTTP 狀態碼是 4xx/5xx 會丟例外（讓 try 失敗）
        tables = pd.read_html(io.StringIO(resp.text)) # 把「字串包裝成一個像檔案的物件（file-like object）」can directly pd.read_html(resp.text)
        df_sp500 = pick_table_by_columns(tables, {"Symbol"})
        return df_sp500["Symbol"].tolist() # if no raise or error then return and end the function 
    except Exception as e:
        print(f"[WARN] StockAnalysis failed, fallback to Wikipedia. error={e}") # if use raise = whole function failed, print so goes to wiki part

    tables = pd.read_html(SP500_WIKI)
    df_sp500 = tables[0]
    if "Symbol" not in df_sp500.columns:
        raise ValueError("Symbol column not found in Wikipedia table")
    return df_sp500["Symbol"].tolist()

def get_sp500_table_from_web():
    """抓 S&P500 full table（含 Market Cap）"""
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(SP500_URL, headers=headers, timeout=15)
    resp.raise_for_status()
    tables = pd.read_html(io.StringIO(resp.text))
    df_sp500 = pick_table_by_columns(tables, {"Symbol"})
    return df_sp500

def get_symbols_table(engine):
    """建立 stocks table（若不存在）"""
    metadata = MetaData()
    table = Table(
        "stocks", metadata,
        Column("symbol", String, primary_key=True),
        Column("status", Boolean, nullable=False),
        Column("date_added", Date),
        Column("date_removed", Date),
    )
    metadata.create_all(engine)
    return table

def update_symbols():
    """
    同步 S&P500（每日跑一次），直接使用網站回傳的 symbol
    若網站失敗 -> 使用 DB 現有 symbols
    """
    engine = get_engine()
    table = get_symbols_table(engine)
    today = datetime.date.today()

    try:
        raw_symbols = get_sp500_symbols_from_web()
        canonical_set = set(raw_symbols)
    except Exception as e:
        print(f"[WARN] Failed to fetch symbols from web, using DB cache. error={e}")
        with engine.connect() as conn: # 讀 DB 現有 symbols 當備援
            rows = conn.execute(select(table.c.symbol)).fetchall()
        canonical_set = {r[0] for r in rows} # {} -> set if not K:V inside {}

    with engine.begin() as conn: # 開一個 transaction（交易）；區塊成功自動 commit，出錯自動 rollback。
        rows = conn.execute(select(table.c.symbol)).fetchall()
        existing_symbols = {r[0] for r in rows}

        for sym in canonical_set: # all in updated symbol list
            if sym not in existing_symbols: #find the new symbol that not in original db and insert
                conn.execute(table.insert().values(symbol=sym, status=True, date_added=today))
            else:
                row = conn.execute(select(table).where(table.c.symbol == sym)).fetchone() # only select one
                if row and not row._mapping["status"]: # have data fetched and status is false 
                    conn.execute(
                        update(table)
                        .where(table.c.symbol == sym)
                        .values(status=True, date_added=today, date_removed=None)
                    )

        for sym in list(existing_symbols): # all in table which is not in sp500 list and turn to inactive 
            if sym not in canonical_set:
                row = conn.execute(select(table).where(table.c.symbol == sym)).fetchone()
                if row and row._mapping["status"]:
                    conn.execute(
                        update(table)
                        .where(table.c.symbol == sym)
                        .values(status=False, date_removed=today)
                    )

    print("update_symbols done. total symbols:", len(canonical_set))

def get_active_symbols() -> List[str]:
    """回傳目前 status=True 的 symbol list"""
    engine = get_engine()
    metadata = MetaData()
    table = Table("stocks", metadata, autoload_with=engine) #去資料庫「找現有的 sp500_symbols 表」，把欄位結構讀回來（reflection），讓你不用再手寫 Column。
    with engine.connect() as conn:
        rows = conn.execute(select(table).where(table.c.status == True)).fetchall()
    return [r._mapping["symbol"] for r in rows]




if __name__ == "__main__":
    update_symbols()

update_symbols done. total symbols: 503
